# AstroPlanner — Planner Prompt Evaluation

This notebook develops and evaluates the prompt for the AstroPlanner imaging plan generator.
Given a DSO target and equipment configuration(s), the prompt should:
1. Recommend which equipment configuration to use
2. Recommend which filters to use
3. Recommend exposure times for each filter

**Workflow:** Edit the system prompt or user prompt template → run Section 6 → review graded results in Section 9.

## 1. Setup

In [1]:
import os
import re
import math
import json
import pathlib
from dataclasses import dataclass, field
from typing import Optional
from dotenv import load_dotenv
import anthropic
import pandas as pd
from IPython.display import display, Markdown

load_dotenv()  # loads ANTHROPIC_API_KEY from .env

client = anthropic.Anthropic()  # picks up ANTHROPIC_API_KEY automatically
EVAL_MODEL = "claude-sonnet-4-6"   # model used to call the prompt under test
JUDGE_MODEL = "claude-sonnet-4-6"  # model used for LLM-as-judge grading
print("Anthropic client ready")

Anthropic client ready


## 2. Data Models

In [2]:
@dataclass
class CelestialObject:
    object_id: str
    display_name: str
    ra: float                          # decimal degrees
    dec: float                         # decimal degrees
    obj_type: str                      # galaxy, nebula, cluster, etc.
    sub_type: Optional[str] = None     # e.g. "Emission Nebula", "Spiral Galaxy"
    constellation: Optional[str] = None
    magnitude: Optional[float] = None
    angular_size_major: Optional[float] = None  # arcminutes
    angular_size_minor: Optional[float] = None  # arcminutes


@dataclass
class EquipmentConfig:
    name: str                    # user-assigned name
    ota_name: str                # OTA model
    camera_name: str             # camera model
    focal_length: float          # mm — native telescope focal length
    aperture: float              # mm — telescope aperture diameter
    focal_reducer_factor: float  # 1.0 = no reducer
    pixel_size: float            # µm — physical pixel size
    resolution_width: int        # sensor width in pixels
    resolution_height: int       # sensor height in pixels
    is_mono: bool = False        # True = monochrome camera


@dataclass
class Filter:
    name: str                           # must match exactly what appears in prompts
    filter_type: str                    # "narrowband" | "broadband"
    is_uv_ir_cut: bool = False          # True = qualifies as narrowband companion
    target_wavelengths: list = field(default_factory=list)  # e.g. ["Ha", "OIII"]
    bandwidth_nm: Optional[float] = None  # passband width; None for broadband
    description: Optional[str] = None


@dataclass
class SiteConditions:
    bortle: int              # 1 (darkest) – 9 (most light polluted)
    altitude_m: float        # metres above sea level
    location_name: str = ""  # optional label


print("Data models defined")

Data models defined


## 3. Equipment Calculations

Matches `buildPrompt()` in `PlanCreationScreen.kt`.

In [3]:
def compute_equipment_stats(cfg: EquipmentConfig) -> dict:
    """Compute derived optical quantities for an EquipmentConfig."""
    effective_fl = cfg.focal_length * cfg.focal_reducer_factor
    f_ratio = effective_fl / cfg.aperture
    sensor_w_mm = cfg.pixel_size * cfg.resolution_width / 1000
    sensor_h_mm = cfg.pixel_size * cfg.resolution_height / 1000
    fov_w = 2 * math.degrees(math.atan(sensor_w_mm / (2 * effective_fl))) * 60  # arcmin
    fov_h = 2 * math.degrees(math.atan(sensor_h_mm / (2 * effective_fl))) * 60  # arcmin
    plate_scale = (cfg.pixel_size / effective_fl) * 206.265  # arcsec/px
    return {
        "effective_fl": round(effective_fl, 1),
        "f_ratio": round(f_ratio, 1),
        "fov_w": round(fov_w, 1),
        "fov_h": round(fov_h, 1),
        "plate_scale": round(plate_scale, 3),
    }


def format_ra(ra_deg: float) -> str:
    """Format RA in decimal degrees as hh h mm m ss.ss s."""
    total_hours = ra_deg / 15.0
    h = int(total_hours)
    m = int((total_hours - h) * 60)
    s = ((total_hours - h) * 60 - m) * 60
    return f"{h}h {m}m {s:.1f}s"


def format_dec(dec_deg: float) -> str:
    """Format Dec in decimal degrees as ±DD°MM'SS"."""
    sign = "+" if dec_deg >= 0 else "-"
    dec_deg = abs(dec_deg)
    d = int(dec_deg)
    m = int((dec_deg - d) * 60)
    s = ((dec_deg - d) * 60 - m) * 60
    return f"{sign}{d:02d}°{m:02d}'{s:04.1f}\""


# Quick sanity check
c8 = EquipmentConfig(
    name="C8 + 0.63x Reducer + ASI294MC Pro",
    ota_name="Celestron C8",
    camera_name="ZWO ASI 294MC Pro",
    focal_length=2032, aperture=203.2, focal_reducer_factor=0.63,
    pixel_size=4.63, resolution_width=4144, resolution_height=2822,
)
redcat = EquipmentConfig(
    name="RedCat 61 + ASI2600MC Duo",
    ota_name="William Optics RedCat 61",
    camera_name="ZWO ASI 2600MC Duo",
    focal_length=300, aperture=61, focal_reducer_factor=1.0,
    pixel_size=3.76, resolution_width=6248, resolution_height=4176,
)

for cfg in [c8, redcat]:
    stats = compute_equipment_stats(cfg)
    print(f"{cfg.name}")
    print(f"  Effective FL: {stats['effective_fl']}mm  f/{stats['f_ratio']}")
    print(f"  FOV: {stats['fov_w']}' × {stats['fov_h']}'   Plate scale: {stats['plate_scale']}\"/px")
    print()

C8 + 0.63x Reducer + ASI294MC Pro
  Effective FL: 1280.2mm  f/6.3
  FOV: 51.5' × 35.1'   Plate scale: 0.746"/px

RedCat 61 + ASI2600MC Duo
  Effective FL: 300.0mm  f/4.9
  FOV: 269.1' × 179.9'   Plate scale: 2.585"/px



## 4. Filter Inventory

In [4]:
# Default filter inventory — edit as needed
DEFAULT_FILTERS = [
    Filter(
        name="Optolong L-eXtreme",
        filter_type="narrowband",
        is_uv_ir_cut=False,
        target_wavelengths=["Ha", "OIII"],
        bandwidth_nm=7,
        description="Dual-band narrowband; passes Hα 656nm and OIII 500nm at 7nm bandwidth",
    ),
    Filter(
        name="Optolong UV/IR Cut",
        filter_type="broadband",
        is_uv_ir_cut=True,
        target_wavelengths=[],
        bandwidth_nm=None,
        description="Broadband UV/IR cut; passes full visible spectrum for natural star colours",
    ),
]

print("Filter inventory:")
for f in DEFAULT_FILTERS:
    wl = ", ".join(f.target_wavelengths) if f.target_wavelengths else "full visible"
    bw = f"{f.bandwidth_nm}nm" if f.bandwidth_nm else "broadband"
    print(f"  {f.name} — {f.filter_type}, {bw}, passes: {wl}")

Filter inventory:
  Optolong L-eXtreme — narrowband, 7nm, passes: Ha, OIII
  Optolong UV/IR Cut — broadband, broadband, passes: full visible


## 5. Prompt Builder

Edit `SYSTEM_PROMPT` or `build_user_prompt()` here to test variations.

In [5]:
SYSTEM_PROMPT = """You are an expert astrophotographer.
Provide a recommended equipment configuration, set of filters and exposure time for each filter for a given target.
Guidelines:
1) The equipment configuration should be selected such that the target best fits the field of view.
2) Before selecting filters, research the specific target. Catalog classifications (galaxy, emission 
   nebula, etc.) are a starting point only — investigate the actual physical characteristics to 
   identify any secondary emission, reflection, or outflow components.
3) Filter selection should be based on the physical light sources present in the target:
   - Ionized gas (Hα, OIII, SII) suggests narrowband
   - Reflection nebula, star clusters or galaxies suggests UV/IR cut (broadband)
   - Some targets have both. Common cases include emission nebulae with reflection components (e.g. M20 and M42),
     starburst galaxies with Hα outflows (e.g. M82), and galaxy fields containing IFN. Where a secondary
     component significant, select filters accordingly and plan to blend.
4) Exposure time should be selected based upon the brightness of the target. Maximum sub-exposure is 300s — modern cooled CMOS sensors have low enough read noise that longer subs offer no SNR benefit and increase the risk of rejected frames.

Report Output — use exactly this markdown structure, no other sections or prose:

## Target Summary
[Two sentences maximum describing the target's physical characteristics and relevant light sources.]

## Recommended Configuration
**[Configuration Name]** — [One sentence: why this FOV fits the target angular size.]

## Recommended Filters

**[Filter Name]** — Sub-exposure: [N]s — [One sentence rationale.]

**[Filter Name]** — Sub-exposure: [N]s — [One sentence rationale.]

Only include filters that are part of the recommendation. Separate each filter with a blank line. Do not include any text outside these three sections.

## Common Mistakes to Avoid

The following are examples of incorrect recommendations. Do not produce output like these.

**M42 (Orion Nebula) — WRONG:**
> ## Recommended Filters
> **Optolong L-eXtreme** — Sub-exposure: 180s — Strong Hα and OIII emission from the ionized nebula makes dual-band narrowband ideal.
> **Optolong UV/IR Cut** — Sub-exposure: 120s — Companion broadband for natural star colours.

Why it is wrong: M42 has a large reflection nebula component surrounding the Trapezium. Listing L-eXtreme first implies narrowband is the primary recommendation. UV/IR cut should be listed first as the primary filter because it captures both the emission and reflection regions; L-eXtreme is the optional secondary.

**M20 (Trifid Nebula) — WRONG:**
> ## Recommended Filters
> **Optolong L-eXtreme** — Sub-exposure: 240s — The emission lobes of M20 emit strongly in Hα and OIII, making narrowband the best choice.

Why it is wrong: M20 has a prominent blue reflection lobe that narrowband filters block entirely. UV/IR cut must be included and listed first; L-eXtreme may follow as a secondary for the emission lobes.

**M31 (Andromeda Galaxy) — WRONG:**
> ## Recommended Filters
> **Optolong L-eXtreme** — Sub-exposure: 300s — Narrowband captures the Hα star-forming regions within the galaxy's spiral arms.

Why it is wrong: M31 is a broadband target. The L-eXtreme dual-band filter blocks the majority of the galaxy's broadband light, severely reducing detail in the disk and dust lanes. UV/IR cut is the correct and only filter for M31.
"""



def build_user_prompt(
    target: CelestialObject,
    equipment_configs: list[EquipmentConfig],
    filters: list[Filter],
    site: SiteConditions,
) -> str:
    lines = []

    # --- Target ---
    lines.append("## Target")
    type_str = target.sub_type if target.sub_type else target.obj_type.capitalize()
    lines.append(f"{target.display_name} — {type_str}")
    if target.constellation:
        lines.append(f"Constellation: {target.constellation}")
    lines.append(f"RA: {format_ra(target.ra)}   Dec: {format_dec(target.dec)}")
    if target.magnitude is not None:
        lines.append(f"• Magnitude: {target.magnitude}")
    if target.angular_size_major is not None:
        size_str = f"{target.angular_size_major}'"
        if target.angular_size_minor is not None:
            size_str += f" × {target.angular_size_minor}'"
        lines.append(f"• Angular size: {size_str}")
    lines.append("")

    # --- Equipment ---
    lines.append("## Available Equipment Configurations")
    for cfg in equipment_configs:
        s = compute_equipment_stats(cfg)
        camera_type = "Mono" if cfg.is_mono else "Color (OSC)"
        lines.append(f"### {cfg.name}")
        lines.append(f"• OTA: {cfg.ota_name}, Focal Length: {cfg.focal_length}mm, f/{s['f_ratio']}")
        lines.append(f"• Camera: {cfg.camera_name} ({camera_type}), {cfg.resolution_width}×{cfg.resolution_height}px @ {cfg.pixel_size}µm")
        lines.append(f"• Effective focal length: {s['effective_fl']}mm (reducer: {cfg.focal_reducer_factor}x)")
        lines.append(f"• Field of view: {s['fov_w']}' × {s['fov_h']}'   Plate scale: {s['plate_scale']}\"/px")
        lines.append("")

    # --- Site ---
    lines.append("## Observing Site")
    if site.location_name:
        lines.append(f"• Location: {site.location_name}")
    lines.append(f"• Altitude: {site.altitude_m}m ASL")
    lines.append(f"• Bortle scale: {site.bortle}/9")
    lines.append("")

    # --- Filters ---
    lines.append("## Session Parameters")
    lines.append("• Goal: Imaging")
    filter_list = ", ".join(f.name for f in filters)
    lines.append(f"• Available filters: {filter_list}")

    return "\n".join(lines)


print("Prompt builder defined")

Prompt builder defined


### Custom Per-Test-Case Graders

Defined before the test suite so they can be referenced in `custom_graders` lists.

In [6]:
def grade_m20_filter_priority(response: str, tc: dict) -> tuple[bool, str]:
    """M20 is a mixed emission+reflection nebula — UV/IR cut must be recommended
    AND must appear before any narrowband filter (i.e. it is the primary filter)."""
    uv_ir_filters = [f for f in tc["filters"] if f.is_uv_ir_cut]
    narrowband_filters = [f for f in tc["filters"] if f.filter_type == "narrowband"]

    uv_mentioned = any(f.name.lower() in response.lower() for f in uv_ir_filters)
    if not uv_mentioned:
        return False, "UV/IR cut not recommended — required as primary filter for mixed emission+reflection nebula"

    nb_mentioned = any(f.name.lower() in response.lower() for f in narrowband_filters)
    if nb_mentioned:
        uv_pos = min(
            (response.lower().find(f.name.lower()) for f in uv_ir_filters if f.name.lower() in response.lower()),
            default=len(response),
        )
        nb_pos = min(
            (response.lower().find(f.name.lower()) for f in narrowband_filters if f.name.lower() in response.lower()),
            default=len(response),
        )
        if nb_pos < uv_pos:
            return False, "Narrowband listed before UV/IR cut — UV/IR cut should be primary for mixed nebula"

    return True, "UV/IR cut present and listed as primary filter"


def grade_m82_narrowband_coverage(response: str, tc: dict) -> tuple[bool, str]:
    """M82 has prominent Ha outflow filaments — the response should recommend
    a narrowband filter or explicitly acknowledge the Ha structure."""
    narrowband_filters = [f for f in tc["filters"] if f.filter_type == "narrowband"]
    nb_mentioned = any(f.name.lower() in response.lower() for f in narrowband_filters)
    ha_mentioned = bool(re.search(r"\bh[aα]\b|\bhalpha\b|\bhydrogen.?alpha\b|\bfilament", response, re.IGNORECASE))

    if nb_mentioned or ha_mentioned:
        return True, "Narrowband filter or Ha/filament acknowledgement present"
    return False, "No narrowband filter or Ha mention — M82's prominent Ha outflow filaments warrant L-eXtreme"


print("Custom graders defined")

Custom graders defined


## 6. Test Suite

Each test case has:
- `target` — a `CelestialObject`
- `equipment` — list of `EquipmentConfig` (1 or more)
- `filters` — filter inventory for this session
- `site` — `SiteConditions`
- `notes` — what we expect (used by LLM-as-judge, not for hard grading)

In [7]:
import json
import pathlib

def load_dso(object_id: str, dso_path: str = "mobile/dso.json") -> CelestialObject:
    """Load a CelestialObject from dso.json by objectId."""
    with open(pathlib.Path(dso_path)) as f:
        data = json.load(f)
    for d in data:
        if d["objectId"] == object_id:
            return CelestialObject(
                object_id=d["objectId"],
                display_name=d["displayName"],
                ra=d["ra"],
                dec=d["dec"],
                obj_type=d["type"],
                sub_type=d.get("subType"),
                constellation=d.get("constellation"),
                magnitude=d.get("magnitude"),
                angular_size_major=d.get("angularSizeMajor"),
                angular_size_minor=d.get("angularSizeMinor"),
            )
    raise ValueError(f"objectId '{object_id}' not found in {dso_path}")


# Shared site conditions
DEFAULT_SITE = SiteConditions(bortle=2, altitude_m=300, location_name="My Backyard")
BOTH_CONFIGS = [c8, redcat]
FILTERS = DEFAULT_FILTERS

TEST_CASES = [
    {
        "id": "TC01",
        "description": "M42 — large bright emission nebula (short exposures expected)",
        "target": load_dso("m42"),
        "equipment": BOTH_CONFIGS,
        "filters": FILTERS,
        "site": DEFAULT_SITE,
        "notes": (
            "RedCat (FOV ~187'x125') should be recommended — M42 at 90'x60' fits well. "
            "C8 FOV ~52'x35' is too narrow to frame the full nebula. "
            "M42 contains a significant reflection nebula component surrounding the Trapezium — "
            "UV/IR cut should be primary to capture both the emission and reflection regions. "
            "Narrowband may optionally follow but should not be listed first."
        ),
        "custom_graders": [("M42 broadband primary", grade_m20_filter_priority)],
    },
    {
        "id": "TC02",
        "description": "M20 — mixed emission + reflection nebula (Trifid)",
        "target": load_dso("m20"),
        "equipment": BOTH_CONFIGS,
        "filters": FILTERS,
        "site": DEFAULT_SITE,
        "notes": (
            "M20 (Trifid Nebula) at 28'x28' fits in C8 FOV (~52'x35') with room; "
            "RedCat would frame it but the target would be small. C8 preferred for detail. "
            "Mixed nebula: the emission (red) lobe benefits from L-eXtreme; the reflection (blue) lobe "
            "requires broadband UV/IR cut to capture. Expect both filters recommended. "
            "dso.json subType is Emission Nebula — model should ideally recognise the reflection component too."
        ),
        "custom_graders": [("M20 filter priority", grade_m20_filter_priority)],
    },
    {
        "id": "TC03",
        "description": "M82 — starburst galaxy with prominent Ha filaments",
        "target": load_dso("m82"),
        "equipment": BOTH_CONFIGS,
        "filters": FILTERS,
        "site": DEFAULT_SITE,
        "notes": (
            "C8 (FOV ~52'x35') is the right choice — M82 at 11'x5' is a good fit; "
            "RedCat would make it tiny. "
            "Primarily a galaxy so UV/IR cut is valid, but M82 has strong Ha outflow filaments — "
            "L-eXtreme can capture that structure. Expect both filters or at least acknowledgement "
            "of the Ha region. Longer sub-exposures appropriate for a mag 8.3 target."
        ),
        "custom_graders": [("M82 Ha filament coverage", grade_m82_narrowband_coverage)],
    },
    {
        "id": "TC04",
        "description": "M31 — extremely large spiral galaxy (wide-field broadband)",
        "target": load_dso("m31"),
        "equipment": BOTH_CONFIGS,
        "filters": FILTERS,
        "site": DEFAULT_SITE,
        "notes": (
            "RedCat (FOV ~187'x125') is the only config that can approach framing M31 (177'x70'). "
            "C8 FOV ~52'x35' captures only a small portion. "
            "Standard spiral galaxy: UV/IR cut expected. L-eXtreme not well suited (blocks most galaxy light). "
            "Bright target (mag 3.44) — shorter sub-exposures expected."
        ),
    },
]

print(f"{len(TEST_CASES)} test cases loaded from dso.json")
for tc in TEST_CASES:
    obj = tc["target"]
    size = f"{obj.angular_size_major}'" if obj.angular_size_major else "unknown size"
    configs = ", ".join(c.name for c in tc["equipment"])
    print(f"  {tc['id']}: {obj.display_name} ({size})  |  {configs}")

4 test cases loaded from dso.json
  TC01: Orion Nebula (90.0')  |  C8 + 0.63x Reducer + ASI294MC Pro, RedCat 61 + ASI2600MC Duo
  TC02: Trifid Nebula (28.0')  |  C8 + 0.63x Reducer + ASI294MC Pro, RedCat 61 + ASI2600MC Duo
  TC03: Cigar Galaxy (10.99')  |  C8 + 0.63x Reducer + ASI294MC Pro, RedCat 61 + ASI2600MC Duo
  TC04: Andromeda Galaxy (177.83')  |  C8 + 0.63x Reducer + ASI294MC Pro, RedCat 61 + ASI2600MC Duo


## 7. Preview a Prompt

Inspect the formatted prompt for any test case before running the full eval.

In [8]:
PREVIEW_TC = TEST_CASES[0]  # change index to preview a different case

prompt = build_user_prompt(
    target=PREVIEW_TC["target"],
    equipment_configs=PREVIEW_TC["equipment"],
    filters=PREVIEW_TC["filters"],
    site=PREVIEW_TC["site"],
)
print("=== SYSTEM PROMPT ===")
print(SYSTEM_PROMPT)
print("\n=== USER PROMPT ===")
print(prompt)

=== SYSTEM PROMPT ===
You are an expert astrophotographer.
Provide a recommended equipment configuration, set of filters and exposure time for each filter for a given target.
Guidelines:
1) The equipment configuration should be selected such that the target best fits the field of view.
2) Before selecting filters, research the specific target. Catalog classifications (galaxy, emission 
   nebula, etc.) are a starting point only — investigate the actual physical characteristics to 
   identify any secondary emission, reflection, or outflow components.
3) Filter selection should be based on the physical light sources present in the target:
   - Ionized gas (Hα, OIII, SII) suggests narrowband
   - Reflection nebula, star clusters or galaxies suggests UV/IR cut (broadband)
   - Some targets have both. Common cases include emission nebulae with reflection components (e.g. M20 and M42),
     starburst galaxies with Hα outflows (e.g. M82), and galaxy fields containing IFN. Where a secondary
 

## 8. Run Evaluation

Calls the API for each test case and stores the raw responses.

In [9]:
def run_test_case(tc: dict, model: str = EVAL_MODEL) -> str:
    prompt = build_user_prompt(
        target=tc["target"],
        equipment_configs=tc["equipment"],
        filters=tc["filters"],
        site=tc["site"],
    )
    message = client.messages.create(
        model=model,
        max_tokens=1024,
        system=SYSTEM_PROMPT,
        messages=[{"role": "user", "content": prompt}],
    )
    return message.content[0].text


# Run all test cases
results = []
for tc in TEST_CASES:
    print(f"Running {tc['id']}: {tc['description']}...", end=" ", flush=True)
    response = run_test_case(tc)
    results.append({"tc": tc, "response": response})
    print("done")

print("\nAll test cases complete.")

Running TC01: M42 — large bright emission nebula (short exposures expected)... done
Running TC02: M20 — mixed emission + reflection nebula (Trifid)... done
Running TC03: M82 — starburst galaxy with prominent Ha filaments... done
Running TC04: M31 — extremely large spiral galaxy (wide-field broadband)... done

All test cases complete.


## 9. Graders

### 9a. Programmatic Graders

These check objective criteria that don't require judgment.

In [10]:
def grade_equipment_named(response: str, equipment: list[EquipmentConfig]) -> tuple[bool, str]:
    """At least one equipment config name appears in the response."""
    found = [cfg.name for cfg in equipment if cfg.name.lower() in response.lower()]
    if found:
        return True, f"Found: {found[0]}"
    return False, "No equipment config name found in response"


def grade_filter_names_valid(response: str, filters: list[Filter]) -> tuple[bool, str]:
    """At least one filter from the inventory is mentioned in the response."""
    mentioned = [f.name for f in filters if f.name.lower() in response.lower()]
    if not mentioned:
        return False, "No filter names from inventory found in response"
    return True, f"Mentioned: {mentioned}"


def grade_exposure_time_present(response: str) -> tuple[bool, str]:
    """At least one exposure time in the expected 'Sub-exposure: Ns' format is present."""
    # Primary: check for the standardised format
    primary = re.search(r'Sub-exposure:\s*\d+\s*s\b', response, re.IGNORECASE)
    if primary:
        return True, f"Found: '{primary.group(0)}'"
    # Fallback: any numeric time value
    fallback_patterns = [
        r'\b(\d+)\s*s\b',
        r'\b(\d+)\s*seconds?\b',
        r'\b(\d+)\s*-\s*second\b',
        r'\b(\d+)\s*min\b',
    ]
    for pat in fallback_patterns:
        match = re.search(pat, response, re.IGNORECASE)
        if match:
            return True, f"Found (non-standard format): '{match.group(0)}'"
    return False, "No exposure time found — expected 'Sub-exposure: Ns' format"


def grade_narrowband_companion(response: str, filters: list[Filter]) -> tuple[bool, str]:
    """If a narrowband filter is recommended, a UV/IR cut companion is also included."""
    narrowband_filters = [f for f in filters if f.filter_type == "narrowband"]
    uv_ir_filters = [f for f in filters if f.is_uv_ir_cut]

    nb_mentioned = any(f.name.lower() in response.lower() for f in narrowband_filters)
    uv_mentioned = any(f.name.lower() in response.lower() for f in uv_ir_filters)

    if not nb_mentioned:
        return True, "No narrowband filter recommended — rule N/A"
    if nb_mentioned and uv_mentioned:
        return True, "Narrowband + UV/IR cut companion both present"
    return False, "Narrowband recommended but no UV/IR cut companion found"


def grade_no_hallucinated_gear(response: str, equipment: list[EquipmentConfig], filters: list[Filter]) -> tuple[bool, str]:
    """Response does not mention gear that wasn't in the prompt."""
    provided_terms = set()
    for cfg in equipment:
        provided_terms.update(cfg.name.lower().split())
        provided_terms.update(cfg.ota_name.lower().split())
        provided_terms.update(cfg.camera_name.lower().split())
    for f in filters:
        provided_terms.update(f.name.lower().split())

    suspect_brands = ["skywatcher", "takahashi", "televue", "explore scientific",
                      "idas", "astronomik", "baader", "chroma", "antlia"]
    found_hallucinations = [
        b for b in suspect_brands
        if b in response.lower() and b not in " ".join(provided_terms).lower()
    ]
    if found_hallucinations:
        return False, f"Possible hallucinated gear: {found_hallucinations}"
    return True, "No obvious hallucinated gear detected"


def grade_report_structure(response: str) -> tuple[bool, str]:
    """Response contains all three required sections with exact ## headers."""
    has_target  = bool(re.search(r'^##\s*Target Summary', response, re.IGNORECASE | re.MULTILINE))
    has_config  = bool(re.search(r'^##\s*Recommended Configuration', response, re.IGNORECASE | re.MULTILINE))
    has_filters = bool(re.search(r'^##\s*Recommended Filters', response, re.IGNORECASE | re.MULTILINE))

    present = [s for s, f in [("Target Summary", has_target), ("Recommended Configuration", has_config), ("Recommended Filters", has_filters)] if f]
    missing = [s for s, f in [("Target Summary", has_target), ("Recommended Configuration", has_config), ("Recommended Filters", has_filters)] if not f]

    if not missing:
        return True, f"All three sections present"
    return False, f"Missing sections: {missing}"


PROGRAMMATIC_GRADES = [
    ("Equipment named",          lambda r, tc: grade_equipment_named(r, tc["equipment"])),
    ("Filter names valid",       lambda r, tc: grade_filter_names_valid(r, tc["filters"])),
    ("Exposure time present",    lambda r, tc: grade_exposure_time_present(r)),
    ("Narrowband companion rule",lambda r, tc: grade_narrowband_companion(r, tc["filters"])),
    ("No hallucinated gear",     lambda r, tc: grade_no_hallucinated_gear(r, tc["equipment"], tc["filters"])),
    ("Report structure",         lambda r, tc: grade_report_structure(r)),
]

print(f"{len(PROGRAMMATIC_GRADES)} programmatic graders defined")

6 programmatic graders defined


### 9b. LLM-as-Judge Grader

Evaluates criteria that require judgment: equipment justification quality and conciseness.

In [11]:
JUDGE_SYSTEM = """You are evaluating an astrophotography imaging plan generated by an AI assistant.
You will be given the original prompt, the AI's response, expected behaviour notes, and the results
of automated criterion checks. Your job is to evaluate three qualitative criteria and write a concise
overall reasoning paragraph summarising the response quality.
Output only the requested JSON."""


def llm_judge(user_prompt: str, response: str, tc_notes: str, programmatic_results: dict) -> dict:
    auto_summary = "\n".join(
        f"  - {criterion}: {'PASS' if passed else 'FAIL'} — {detail}"
        for criterion, (passed, detail) in programmatic_results.items()
    )
    judge_prompt = f"""## Original Prompt
{user_prompt}

## AI Response to Evaluate
{response}

## Expected Behaviour Notes
{tc_notes}

## Automated Check Results (already computed)
{auto_summary}

## Your Evaluation Tasks

The prompt requires a structured three-section report using exactly these markdown headers:

  ## Target Summary
  [Two sentences maximum describing the target's physical characteristics and light sources.]

  ## Recommended Configuration
  **[Configuration Name]** — [One sentence: why this FOV fits the target angular size.]

  ## Recommended Filters

  **[Filter Name]** — Sub-exposure: [N]s — [One sentence rationale.]

  **[Filter Name]** — Sub-exposure: [N]s — [One sentence rationale.]

  (Each filter on its own line, separated by a blank line.)

Filter selection rules:
  - Ionized gas (Hα, OIII, SII) → narrowband
  - Reflection nebula, star clusters, galaxies → UV/IR cut
  - Mixed targets: M20 and M42 should prioritise broadband (significant reflection components);
    M82 warrants narrowband alongside UV/IR cut (Hα outflows)

Evaluate the following three criteria:

1. **Equipment correct** — Based on the FOV data in the prompt and the target's angular size, was the
   right equipment configuration recommended? PASS if the config whose FOV best fits the target was
   chosen. FAIL if a clearly worse-fitting config was picked.

2. **Physical filter reasoning** — Does the target summary and filter rationale demonstrate reasoning
   from the target's actual physical light sources, not just its catalog label? For mixed targets
   (M20, M42, M82) the model has been explicitly guided — FAIL if it ignores the secondary component.

3. **Report format** — Does the response follow the exact three-section structure?
   PASS requires: (a) all three ## headers present with correct names, (b) target summary ≤2 sentences,
   (c) configuration line starts with bold config name followed by em-dash and one sentence,
   (d) each filter is on its own line (blank line between filters), starts with bold filter name,
   then Sub-exposure: Ns, then em-dash, then one rationale sentence — in that order,
   (e) no prose outside the three sections.
   When counting sentences, a sentence boundary is a period followed by a capital letter beginning
   a new independent clause. Do NOT count periods inside numbers (f/6.3, 90.5'), abbreviations,
   or units as sentence breaks. Two long sentences should be counted as 2, not more.
   FAIL if headers are renamed, sections are merged, sentence limits are clearly exceeded (3+
   distinct sentences), the Sub-exposure format is missing, filters run together on one line,
   or extra sections/preamble are present.

4. **Overall reasoning** — 2–4 sentences summarising quality, what was done well, and any notable
   failures (wrong filter, missing Sub-exposure format, wrong equipment, ignored secondary components,
   incorrect section structure).

Respond with JSON only:
{{
  "equipment_correct": {{"result": "PASS" or "FAIL", "reason": "..."}},
  "physical_filter_reasoning": {{"result": "PASS" or "FAIL", "reason": "..."}},
  "report_format": {{"result": "PASS" or "FAIL", "reason": "..."}},
  "overall_reasoning": "..."
}}"""

    message = client.messages.create(
        model=JUDGE_MODEL,
        max_tokens=1024,
        system=JUDGE_SYSTEM,
        messages=[{"role": "user", "content": judge_prompt}],
    )
    raw = message.content[0].text.strip()
    raw = re.sub(r'^```json\s*', '', raw)
    raw = re.sub(r'\s*```$', '', raw)
    try:
        return json.loads(raw)
    except json.JSONDecodeError as e:
        raise ValueError(f"Judge returned invalid JSON ({e}):\n{raw}") from e


print("LLM-as-judge grader defined")

LLM-as-judge grader defined


## 10. Grade All Results

In [12]:
graded = []

for result in results:
    tc = result["tc"]
    response = result["response"]
    row = {"tc_id": tc["id"], "description": tc["description"], "response": response}

    # Programmatic grades
    prog_results = {}
    for criterion, fn in PROGRAMMATIC_GRADES:
        passed, detail = fn(response, tc)
        prog_results[criterion] = (passed, detail)

    # Custom per-test-case graders
    for criterion, fn in tc.get("custom_graders", []):
        passed, detail = fn(response, tc)
        prog_results[criterion] = (passed, detail)

    # LLM-as-judge (passes programmatic results for context)
    user_prompt = build_user_prompt(
        target=tc["target"],
        equipment_configs=tc["equipment"],
        filters=tc["filters"],
        site=tc["site"],
    )
    judge_result = llm_judge(user_prompt, response, tc["notes"], prog_results)

    # Merge all criteria
    all_criteria = dict(prog_results)
    for key, val in judge_result.items():
        if key == "overall_reasoning":
            row["overall_reasoning"] = val
            continue
        label = key.replace("_", " ").title()
        all_criteria[label] = (val["result"] == "PASS", val["reason"])

    row["criteria"] = all_criteria

    # Score: 1 point per criterion, scaled to 1–10
    n = len(all_criteria)
    passes = sum(1 for passed, _ in all_criteria.values() if passed)
    row["score"] = round((passes / n) * 10, 1)

    row["prompt_inputs"] = user_prompt
    row["notes"] = tc["notes"]
    graded.append(row)
    print(f"{tc['id']} graded  ({passes}/{n} criteria passed  →  score {row['score']})")

print("\nGrading complete.")

TC01 graded  (9/10 criteria passed  →  score 9.0)
TC02 graded  (10/10 criteria passed  →  score 10.0)
TC03 graded  (10/10 criteria passed  →  score 10.0)
TC04 graded  (9/9 criteria passed  →  score 10.0)

Grading complete.


## 11. Results Summary

In [13]:
from IPython.display import HTML

scores = [r["score"] for r in graded]
avg_score = round(sum(scores) / len(scores), 1)
pass_rate = f"{sum(1 for s in scores if s >= 7) / len(scores):.0%}"

# Union of all criteria across all test cases (custom graders vary per test case)
all_criteria = {}  # criterion -> list of (passed, detail) for rows that have it
for r in graded:
    for c, result in r["criteria"].items():
        all_criteria.setdefault(c, []).append(result)

print(f"Total test cases : {len(graded)}")
print(f"Average score    : {avg_score} / 10")
print(f"Pass rate (≥7)   : {pass_rate}")
print()
print("Per-criterion pass rates (n = number of test cases with this criterion):")
for c, results in all_criteria.items():
    n = len(results)
    passes = sum(1 for passed, _ in results if passed)
    rate = passes / n
    bar = "█" * int(rate * 10) + "░" * (10 - int(rate * 10))
    label = f"{c} (n={n})" if n < len(graded) else c
    print(f"  {label:<45} {bar} {rate:.0%}")

Total test cases : 4
Average score    : 9.8 / 10
Pass rate (≥7)   : 100%

Per-criterion pass rates (n = number of test cases with this criterion):
  Equipment named                               ██████████ 100%
  Filter names valid                            ██████████ 100%
  Exposure time present                         ██████████ 100%
  Narrowband companion rule                     ██████████ 100%
  No hallucinated gear                          ██████████ 100%
  Report structure                              ██████████ 100%
  M42 broadband primary (n=1)                   ██████████ 100%
  Equipment Correct                             ██████████ 100%
  Physical Filter Reasoning                     ██████████ 100%
  Report Format                                 ███████░░░ 75%
  M20 filter priority (n=1)                     ██████████ 100%
  M82 Ha filament coverage (n=1)                ██████████ 100%


## 12. HTML Evaluation Report

In [14]:
import html as html_lib
import markdown as md_lib

def build_html_report(graded: list, output_path: str = "planner_eval_report.html") -> str:
    def score_class(s):
        if s >= 8: return "score-high"
        if s >= 5: return "score-medium"
        return "score-low"

    def fmt_prompt_inputs(row):
        # Parse key lines from the stored prompt string
        lines = row["prompt_inputs"].split("\n")
        target_lines, equip_lines, site_lines, filter_lines = [], [], [], []
        section = None
        for line in lines:
            if line.startswith("## Target"): section = "target"
            elif line.startswith("## Available Equipment"): section = "equip"
            elif line.startswith("## Observing Site"): section = "site"
            elif line.startswith("## Session"): section = "filters"
            elif line.strip() and section == "target": target_lines.append(line.strip())
            elif line.strip() and section == "equip": equip_lines.append(line.strip())
            elif line.strip() and section == "site": site_lines.append(line.strip())
            elif line.strip() and section == "filters": filter_lines.append(line.strip())
        parts = []
        for l in target_lines[:4]:
            parts.append(f"<strong>{html_lib.escape(l)}</strong>" if l == target_lines[0] else html_lib.escape(l))
            parts.append("<br>")
        parts.append("<em>Equipment:</em><br>")
        for l in equip_lines:
            if l.startswith("###"): parts.append(f"<strong>{html_lib.escape(l[3:].strip())}</strong><br>")
            elif l.startswith("•"): parts.append(f"&nbsp;&nbsp;{html_lib.escape(l)}<br>")
        parts.append("<em>Site &amp; Filters:</em><br>")
        for l in site_lines + filter_lines:
            if l.startswith("•"): parts.append(f"{html_lib.escape(l)}<br>")
        return "".join(parts)

    def fmt_criteria(notes):
        # Format notes as bullet points
        sentences = [s.strip() for s in notes.replace(". ", ".\n").split("\n") if s.strip()]
        items = "".join(f"<li>{html_lib.escape(s)}</li>" for s in sentences)
        return f"<ul>{items}</ul>"

    def fmt_criteria_detail(criteria):
        rows = ""
        for name, (passed, detail) in criteria.items():
            icon = "✓" if passed else "✗"
            color = "#2e7d32" if passed else "#c62828"
            rows += (
                f"<tr><td style='color:{color};font-weight:bold;white-space:nowrap'>{icon} {html_lib.escape(name)}</td>"
                f"<td style='color:#555;font-size:12px'>{html_lib.escape(detail)}</td></tr>"
            )
        return f"<table style='border-collapse:collapse;font-size:12px;width:100%'>{rows}</table>"

    def fmt_response(text):
        rendered = md_lib.markdown(text, extensions=["fenced_code", "nl2br"])
        return f"<div class='output'>{rendered}</div>"

    rows_html = ""
    for row in graded:
        sc = row["score"]
        rows_html += f"""
        <tr>
            <td><strong>{html_lib.escape(row['tc_id'])}</strong><br>{html_lib.escape(row['description'])}</td>
            <td class="prompt-inputs">{fmt_prompt_inputs(row)}</td>
            <td class="criteria">{fmt_criteria(row['notes'])}</td>
            <td class="output">{fmt_response(row['response'])}</td>
            <td class="score-col"><span class="score {score_class(sc)}">{sc}</span>
                <div style="margin-top:8px">{fmt_criteria_detail(row['criteria'])}</div>
            </td>
            <td>{html_lib.escape(row.get('overall_reasoning', ''))}</td>
        </tr>"""

    avg = round(sum(r["score"] for r in graded) / len(graded), 1)
    pass_count = sum(1 for r in graded if r["score"] >= 7)
    pass_pct = f"{pass_count / len(graded):.0%}"

    html = f"""<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>AstroPlanner — Prompt Evaluation Report</title>
    <style>
        body {{ font-family: Arial, sans-serif; line-height: 1.6; margin: 0; padding: 20px; color: #333; }}
        .header {{ background-color: #1a1a2e; color: #e0e0e0; padding: 20px; border-radius: 5px; margin-bottom: 20px; }}
        .header h1 {{ margin: 0 0 15px 0; color: #a0c4ff; }}
        .summary-stats {{ display: flex; justify-content: flex-start; flex-wrap: wrap; gap: 10px; }}
        .stat-box {{ background-color: #16213e; border-radius: 5px; padding: 15px; box-shadow: 0 2px 5px rgba(0,0,0,0.3);
                     min-width: 160px; }}
        .stat-label {{ font-size: 13px; color: #aaa; }}
        .stat-value {{ font-size: 28px; font-weight: bold; color: #a0c4ff; margin-top: 4px; }}
        table {{ width: 100%; border-collapse: collapse; margin-top: 20px; }}
        th {{ background-color: #2c2c54; color: #e0e0e0; text-align: left; padding: 12px; }}
        td {{ padding: 10px; border-bottom: 1px solid #ddd; vertical-align: top; width: 16%; }}
        tr:nth-child(even) {{ background-color: #f9f9f9; }}
        .score {{ font-weight: bold; font-size: 20px; padding: 6px 12px; border-radius: 3px; display: inline-block; }}
        .score-high {{ background-color: #c8e6c9; color: #2e7d32; }}
        .score-medium {{ background-color: #fff9c4; color: #f57f17; }}
        .score-low {{ background-color: #ffcdd2; color: #c62828; }}
        .output {{ overflow: auto; font-size: 13px; }}
        .output pre {{ background: #f5f5f5; border: 1px solid #ddd; border-radius: 4px; padding: 8px;
                       font-size: 12px; line-height: 1.4; white-space: pre-wrap; word-wrap: break-word; }}
        .output h1, .output h2, .output h3 {{ margin: 6px 0; font-size: 14px; }}
        .output p {{ margin: 4px 0; }}
        .output ul, .output ol {{ margin: 4px 0; padding-left: 18px; }}
        .criteria ul {{ margin: 0; padding-left: 16px; font-size: 13px; }}
        .criteria li {{ margin-bottom: 4px; }}
        .prompt-inputs {{ font-size: 13px; }}
        .score-col {{ width: 200px; }}
    </style>
</head>
<body>
    <div class="header">
        <h1>AstroPlanner — Prompt Evaluation Report</h1>
        <div class="summary-stats">
            <div class="stat-box"><div class="stat-label">Total Test Cases</div><div class="stat-value">{len(graded)}</div></div>
            <div class="stat-box"><div class="stat-label">Average Score</div><div class="stat-value">{avg} / 10</div></div>
            <div class="stat-box"><div class="stat-label">Pass Rate (≥7)</div><div class="stat-value">{pass_pct}</div></div>
        </div>
    </div>
    <table>
        <thead>
            <tr>
                <th>Scenario</th>
                <th>Prompt Inputs</th>
                <th>Solution Criteria</th>
                <th>Output</th>
                <th>Score &amp; Criteria</th>
                <th>Reasoning</th>
            </tr>
        </thead>
        <tbody>{rows_html}</tbody>
    </table>
</body>
</html>"""

    with open(output_path, "w") as f:
        f.write(html)
    print(f"Report saved to {output_path}")
    return html


# Install markdown if needed, then generate report
try:
    import markdown
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "markdown", "-q"])
    import markdown as md_lib

report_html = build_html_report(graded)
display(HTML(report_html))

Report saved to planner_eval_report.html


## 13. Inspect Individual Responses

In [17]:
INSPECT_INDEX = 2  # change index to inspect a different result

r = graded[INSPECT_INDEX]
display(Markdown(f"### {r['tc_id']}: {r['description']}"))
display(Markdown(f"**Notes:** {r['notes']}"))
display(Markdown("---"))
display(Markdown(r["response"]))
print(f"\n--- Criteria (score: {r['score']}/10) ---")
for criterion, (passed, detail) in r["criteria"].items():
    icon = "✓" if passed else "✗"
    print(f"  [{icon}] {criterion}: {detail}")

### TC03: M82 — starburst galaxy with prominent Ha filaments

**Notes:** C8 (FOV ~52'x35') is the right choice — M82 at 11'x5' is a good fit; RedCat would make it tiny. Primarily a galaxy so UV/IR cut is valid, but M82 has strong Ha outflow filaments — L-eXtreme can capture that structure. Expect both filters or at least acknowledgement of the Ha region. Longer sub-exposures appropriate for a mag 8.3 target.

---

## Target Summary
M82 (Cigar Galaxy) is a starburst irregular galaxy with intense star-forming activity driving dramatic Hα-emitting superwind filaments extending perpendicular to the disk, reaching several arcminutes above and below the galactic plane; the disk itself is a broadband source of starlight, dust lanes, and continuum emission requiring UV/IR cut coverage.

## Recommended Configuration
**C8 + 0.63x Reducer + ASI294MC Pro** — The 51.5' × 35.1' field of view comfortably frames M82's 11' × 5' disk while providing enough surrounding space to capture the extended Hα superwind filaments without wasting field on empty sky.

## Recommended Filters

**Optolong UV/IR Cut** — Sub-exposure: 180s — Captures the galaxy's broadband disk, dust lanes, and stellar continuum, which form the primary visual structure of M82.

**Optolong L-eXtreme** — Sub-exposure: 300s — Isolates the extended Hα superwind outflow filaments that are otherwise overwhelmed by the bright galactic disk, enabling a luminance blend to reveal the bipolar outflow structure.


--- Criteria (score: 10.0/10) ---
  [✓] Equipment named: Found: C8 + 0.63x Reducer + ASI294MC Pro
  [✓] Filter names valid: Mentioned: ['Optolong L-eXtreme', 'Optolong UV/IR Cut']
  [✓] Exposure time present: Found: 'Sub-exposure: 180s'
  [✓] Narrowband companion rule: Narrowband + UV/IR cut companion both present
  [✓] No hallucinated gear: No obvious hallucinated gear detected
  [✓] Report structure: All three sections present
  [✓] M82 Ha filament coverage: Narrowband filter or Ha/filament acknowledgement present
  [✓] Equipment Correct: M82 at ~11'×5' fits well in the C8's 51.5'×35.1' FOV, leaving room for the extended Hα superwind filaments. The RedCat 61's 269'×180' FOV would render M82 tiny at roughly 4% of frame height, making the C8 the clear correct choice.
  [✓] Physical Filter Reasoning: The target summary correctly identifies two distinct physical light sources: the broadband stellar/dust disk and the Hα-emitting superwind filaments extending perpendicular to the disk. Bo

In [16]:
results

[(True, 'Narrowband filter or Ha/filament acknowledgement present')]